# Baseline Modeling

This notebook builds the first machine learning benchmark for the AI-driven Loan Analytics Platform.

The goal is not to produce the final best model. The goal is to create a reliable baseline for predicting 12-month loan default using the conservative feature set created in the feature engineering phase.

Scope:

- load the baseline modeling dataset;
- use the feature configuration generated by feature engineering;
- split the data into train and test sets;
- build preprocessing pipelines;
- train baseline classifiers;
- evaluate with default-risk metrics;
- save evaluation reports and the best baseline model artifact.

No hyperparameter tuning is performed here.

## 1. Imports and configuration

We use scikit-learn pipelines so preprocessing happens inside the training workflow. This prevents leakage from imputation or encoding before the train/test split.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_baseline.csv'
FEATURE_CONFIG_PATH = PROJECT_ROOT / 'reports' / 'feature_config_baseline.json'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR = PROJECT_ROOT / 'models'

RESULTS_PATH = REPORTS_DIR / 'baseline_model_results.csv'
CONFUSION_PATH = REPORTS_DIR / 'baseline_confusion_matrices.csv'
CLASSIFICATION_REPORT_PATH = REPORTS_DIR / 'baseline_classification_report.csv'
METADATA_PATH = REPORTS_DIR / 'baseline_modeling_metadata.json'
BEST_MODEL_PATH = MODELS_DIR / 'baseline_best_model.joblib'

RANDOM_STATE = 42
TEST_SIZE = 0.20
HGB_MAX_TRAIN_ROWS = 150_000

print(f'Data path: {DATA_PATH}')
print(f'Feature config path: {FEATURE_CONFIG_PATH}')

Data path: C:\Users\aitmo\financial-loan-analytics\data\processed\modeling_dataset_baseline.csv
Feature config path: C:\Users\aitmo\financial-loan-analytics\reports\feature_config_baseline.json


## 2. Load modeling dataset and feature configuration

The modeling dataset was created by the feature engineering notebook. The JSON config tells this notebook which columns are identifiers, features, categorical features, numerical features, and the target.

In [ ]:
model_df = pd.read_csv(DATA_PATH, low_memory=False)

with open(FEATURE_CONFIG_PATH, encoding='utf-8') as file:
    feature_config = json.load(file)

target_column = feature_config['target_column']
identifier_columns = feature_config['identifier_columns']
feature_columns = feature_config['model_input_features']
numeric_features = feature_config['numeric_features']
categorical_features = feature_config['categorical_features']

print(f'Dataset shape: {model_df.shape}')
print(f'Target column: {target_column}')
print(f'Identifier columns: {identifier_columns}')
print(f'Feature columns: {len(feature_columns)}')
print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

display(model_df.head())

Dataset shape: (563510, 29)
Target column: has_default_within_12_months
Identifier columns: ['loan_id']
Feature columns: 27
Numeric features: 26
Categorical features: 1


,loan_id,has_default_within_12_months,country,issued_amount,initial_interest_rate,nr_of_payments,initial_loan_duration,combined_income,issue_year,issue_month,issue_quarter,issue_day_of_week,issue_month_sin,issue_month_cos,loan_issued_at_missing,log_issued_amount,combined_income_missing,log_combined_income,loan_to_income_ratio,installment_amount_proxy,installment_to_income_proxy,interest_cost_proxy,amount_per_duration_unit,issued_amount_missing,initial_interest_rate_missing,nr_of_payments_missing,initial_loan_duration_missing,loan_to_income_ratio_missing,installment_to_income_proxy_missing
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,0,Finland,310.0000,0.1780,1,12,"2,068.6800",2025,3,1,4,1.0000,0.0000,0,5.7398,0,7.6351,0.1499,310.0000,0.1499,55.1800,25.8333,0,0,0,0,0,0
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,0,Finland,414.0000,0.1780,120,18,"1,469.5000",2025,1,1,4,0.5000,0.8660,0,6.0283,0,7.2934,0.2817,3.4500,0.0023,73.6920,23.0000,0,0,0,0,0,0
2,7362E260-54DC-4798-8848-B18C00A59FCB,0,Finland,518.0000,0.1877,8,60,"4,513.2000",2024,6,2,1,0.0000,-1.0000,0,6.2519,0,8.4150,0.1148,64.7500,0.0143,97.2286,8.6333,0,0,0,0,0,0
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,0,Finland,"3,731.0000",0.1780,75,60,"1,301.0000",2025,1,1,3,0.5000,0.8660,0,8.2247,0,7.1717,2.8678,49.7467,0.0382,664.1180,62.1833,0,0,0,0,0,0
4,BE865034-6041-4855-AB42-B196014FA0E8,0,Netherlands,"3,432.0000",0.1056,38,60,"5,194.7000",2024,6,2,4,0.0000,-1.0000,0,8.1412,0,8.5556,0.6607,90.3158,0.0174,362.4192,57.2000,0,0,0,0,0,0


## 3. Modeling dataset checks

Before training, we verify that the target has no missing values and that all configured feature columns exist.

In [ ]:
missing_config_features = [
    column for column in feature_columns if column not in model_df.columns
]
if missing_config_features:
    raise KeyError(f'Missing configured features: {missing_config_features}')

if target_column not in model_df.columns:
    raise KeyError(f'Target column not found: {target_column}')

if model_df[target_column].isna().sum() > 0:
    raise ValueError('Target column contains missing values.')

modeling_checks = pd.DataFrame([
    {'check': 'rows', 'value': len(model_df)},
    {'check': 'columns', 'value': model_df.shape[1]},
    {'check': 'target_missing_values', 'value': int(model_df[target_column].isna().sum())},
    {'check': 'duplicate_column_names', 'value': int(pd.Index(model_df.columns).duplicated().sum())},
    {'check': 'duplicate_rows', 'value': int(model_df.duplicated().sum())},
])

display(modeling_checks)

class_balance = (
    model_df[target_column]
    .value_counts()
    .sort_index()
    .rename_axis(target_column)
    .reset_index(name='count')
)
class_balance['percentage'] = class_balance['count'] / len(model_df)

display(class_balance)

fig = px.bar(
    class_balance,
    x=target_column,
    y='count',
    text=class_balance['percentage'].map(lambda value: f'{value:.1%}'),
    title='Target Class Distribution',
)
fig.update_traces(textposition='outside')
fig.show()

,check,value
0,rows,563510
1,columns,29
2,target_missing_values,0
3,duplicate_column_names,0
4,duplicate_rows,0


,has_default_within_12_months,count,percentage
0,0,480438,0.8526
1,1,83072,0.1474


## 4. Train/test split

We use a stratified train/test split so the default/non-default class proportions are preserved in both sets.

A later professional improvement may use a time-based split, especially if the final objective is future default prediction.

In [ ]:
X = model_df[feature_columns].copy()
y = model_df[target_column].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(X_train), 'default_rate': y_train.mean()},
    {'split': 'test', 'rows': len(X_test), 'default_rate': y_test.mean()},
])

display(split_summary)

,split,rows,default_rate
0,train,450808,0.1474
1,test,112702,0.1474


## 5. Preprocessing pipelines

The logistic model uses median imputation, scaling for numerical features, and one-hot encoding for categorical features.

The gradient boosting model uses median imputation for numerical features and ordinal encoding for categorical features. This keeps the model efficient for a large dataset.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def make_ordinal_encoder():
    try:
        return OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-1,
        )
    except TypeError:
        return OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        )


numeric_pipeline_linear = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline_one_hot = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', make_one_hot_encoder()),
    ]
)

linear_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline_linear, numeric_features),
        ('categorical', categorical_pipeline_one_hot, categorical_features),
    ],
    remainder='drop',
)

numeric_pipeline_tree = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
    ]
)

categorical_pipeline_ordinal = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', make_ordinal_encoder()),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline_tree, numeric_features),
        ('categorical', categorical_pipeline_ordinal, categorical_features),
    ],
    remainder='drop',
)

print('Preprocessing pipelines created.')

Preprocessing pipelines created.


## 6. Define baseline models

We train three baseline models:

1. Dummy Classifier: naive benchmark.
2. Logistic Regression: interpretable baseline with class balancing.
3. HistGradientBoosting: lightweight non-linear benchmark.

The gradient boosting model is trained on a stratified sample of the training set for speed. It is still evaluated on the full test set.

In [ ]:
models = {
    'dummy_most_frequent': Pipeline(
        steps=[
            ('preprocessor', linear_preprocessor),
            ('model', DummyClassifier(strategy='most_frequent')),
        ]
    ),
    'logistic_regression_balanced': Pipeline(
        steps=[
            ('preprocessor', linear_preprocessor),
            ('model', LogisticRegression(
                max_iter=500,
                class_weight='balanced',
                solver='lbfgs',
                random_state=RANDOM_STATE,
            )),
        ]
    ),
    'hist_gradient_boosting': Pipeline(
        steps=[
            ('preprocessor', tree_preprocessor),
            ('model', HistGradientBoostingClassifier(
                max_iter=80,
                learning_rate=0.08,
                max_leaf_nodes=31,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
}

print(f'Models defined: {list(models)}')

Models defined: ['dummy_most_frequent', 'logistic_regression_balanced', 'hist_gradient_boosting']


## 7. Train baseline models

Training is intentionally simple and reproducible. Hyperparameter tuning will be handled later.

In [ ]:
trained_models = {}
training_rows = []

if len(X_train) > HGB_MAX_TRAIN_ROWS:
    X_train_hgb, _, y_train_hgb, _ = train_test_split(
        X_train,
        y_train,
        train_size=HGB_MAX_TRAIN_ROWS,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_train_hgb = X_train
    y_train_hgb = y_train

for model_name, pipeline in models.items():
    if model_name == 'hist_gradient_boosting':
        X_fit = X_train_hgb
        y_fit = y_train_hgb
    else:
        X_fit = X_train
        y_fit = y_train

    print(f'Training {model_name} on {len(X_fit):,} rows...')
    pipeline.fit(X_fit, y_fit)
    trained_models[model_name] = pipeline
    training_rows.append({'model': model_name, 'training_rows': len(X_fit)})

training_summary = pd.DataFrame(training_rows)
display(training_summary)

Training dummy_most_frequent on 450,808 rows...


Training logistic_regression_balanced on 450,808 rows...


Training hist_gradient_boosting on 150,000 rows...


,model,training_rows
0,dummy_most_frequent,450808
1,logistic_regression_balanced,450808
2,hist_gradient_boosting,150000


## 8. Evaluate models

For default-risk prediction, accuracy alone is not enough. We evaluate:

- precision;
- recall;
- F1-score;
- ROC-AUC;
- PR-AUC / average precision;
- confusion matrix.

Recall and PR-AUC are especially important when the positive class is less frequent.

In [ ]:
def predict_positive_probability(model, features):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1]
    if hasattr(model, 'decision_function'):
        scores = model.decision_function(features)
        return 1 / (1 + np.exp(-scores))
    return None


results = []
confusion_rows = []
classification_rows = []
curves = {}

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = predict_positive_probability(model, X_test)

    result = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
    }

    if y_proba is not None and len(np.unique(y_proba)) > 1:
        result['roc_auc'] = roc_auc_score(y_test, y_proba)
        result['average_precision'] = average_precision_score(y_test, y_proba)
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
        curves[model_name] = {
            'fpr': fpr,
            'tpr': tpr,
            'precision': precision_curve,
            'recall': recall_curve,
        }
    else:
        result['roc_auc'] = np.nan
        result['average_precision'] = np.nan

    results.append(result)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    confusion_rows.extend([
        {'model': model_name, 'actual': 0, 'predicted': 0, 'count': int(cm[0, 0])},
        {'model': model_name, 'actual': 0, 'predicted': 1, 'count': int(cm[0, 1])},
        {'model': model_name, 'actual': 1, 'predicted': 0, 'count': int(cm[1, 0])},
        {'model': model_name, 'actual': 1, 'predicted': 1, 'count': int(cm[1, 1])},
    ])

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            row = {'model': model_name, 'label': label}
            row.update(metrics)
            classification_rows.append(row)

results_df = pd.DataFrame(results).sort_values(
    ['average_precision', 'roc_auc'],
    ascending=False,
)
confusion_df = pd.DataFrame(confusion_rows)
classification_report_df = pd.DataFrame(classification_rows)

display(results_df)
display(confusion_df)
display(classification_report_df.head(20))

,model,accuracy,precision,recall,f1,roc_auc,average_precision
2,hist_gradient_boosting,0.8804,0.7034,0.3259,0.4455,0.8793,0.5896
1,logistic_regression_balanced,0.7300,0.3173,0.7222,0.4409,0.8155,0.4758
0,dummy_most_frequent,0.8526,0.0000,0.0000,0.0000,NaN,NaN


,model,actual,predicted,count
0,dummy_most_frequent,0,0,96088
1,dummy_most_frequent,0,1,0
2,dummy_most_frequent,1,0,16614
3,dummy_most_frequent,1,1,0
4,logistic_regression_balanced,0,0,70272
5,logistic_regression_balanced,0,1,25816
6,logistic_regression_balanced,1,0,4615
7,logistic_regression_balanced,1,1,11999
8,hist_gradient_boosting,0,0,93805
9,hist_gradient_boosting,0,1,2283


,model,label,precision,recall,f1-score,support
0,dummy_most_frequent,0,0.8526,1.0000,0.9204,"96,088.0000"
1,dummy_most_frequent,1,0.0000,0.0000,0.0000,"16,614.0000"
2,dummy_most_frequent,macro avg,0.4263,0.5000,0.4602,"112,702.0000"
3,dummy_most_frequent,weighted avg,0.7269,0.8526,0.7847,"112,702.0000"
4,logistic_regression_balanced,0,0.9384,0.7313,0.8220,"96,088.0000"
5,logistic_regression_balanced,1,0.3173,0.7222,0.4409,"16,614.0000"
6,logistic_regression_balanced,macro avg,0.6278,0.7268,0.6315,"112,702.0000"
7,logistic_regression_balanced,weighted avg,0.8468,0.7300,0.7658,"112,702.0000"
8,hist_gradient_boosting,0,0.8933,0.9762,0.9330,"96,088.0000"
9,hist_gradient_boosting,1,0.7034,0.3259,0.4455,"16,614.0000"


## 9. Visual model comparison

These charts compare model performance. The dummy model should be treated as a floor, not a useful credit-risk model.

In [9]:
metrics_to_plot = [
    'roc_auc',
    'average_precision',
    'recall',
    'precision',
    'f1',
]

plot_results = results_df.melt(
    id_vars='model',
    value_vars=metrics_to_plot,
    var_name='metric',
    value_name='score',
)

fig = px.bar(
    plot_results,
    x='metric',
    y='score',
    color='model',
    barmode='group',
    title='Baseline Model Comparison',
)
fig.show()

## 10. ROC and Precision-Recall curves

Curves help evaluate ranking quality beyond a single threshold.

In [10]:
roc_curve_rows = []
pr_curve_rows = []

for model_name, curve_data in curves.items():
    # Downsample curve points for compact notebook output.
    roc_indices = np.linspace(0, len(curve_data['fpr']) - 1, min(500, len(curve_data['fpr']))).astype(int)
    pr_indices = np.linspace(0, len(curve_data['recall']) - 1, min(500, len(curve_data['recall']))).astype(int)

    roc_curve_rows.extend([
        {
            'model': model_name,
            'false_positive_rate': curve_data['fpr'][index],
            'true_positive_rate': curve_data['tpr'][index],
        }
        for index in roc_indices
    ])

    pr_curve_rows.extend([
        {
            'model': model_name,
            'recall': curve_data['recall'][index],
            'precision': curve_data['precision'][index],
        }
        for index in pr_indices
    ])

roc_curve_df = pd.DataFrame(roc_curve_rows)
pr_curve_df = pd.DataFrame(pr_curve_rows)

if not roc_curve_df.empty:
    fig = px.line(
        roc_curve_df,
        x='false_positive_rate',
        y='true_positive_rate',
        color='model',
        title='ROC Curves',
    )
    fig.show()

if not pr_curve_df.empty:
    fig = px.line(
        pr_curve_df,
        x='recall',
        y='precision',
        color='model',
        title='Precision-Recall Curves',
    )
    fig.show()

## 11. Select and save the best baseline model

For default-risk ranking, PR-AUC / average precision is often more informative than accuracy when the positive class is imbalanced. We select the best baseline model by average precision, then ROC-AUC as a tie-breaker.

In [11]:
best_model_name = results_df.iloc[0]['model']
best_model = trained_models[best_model_name]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(RESULTS_PATH, index=False)
confusion_df.to_csv(CONFUSION_PATH, index=False)
classification_report_df.to_csv(CLASSIFICATION_REPORT_PATH, index=False)
joblib.dump(best_model, BEST_MODEL_PATH)

metadata = {
    'target_column': target_column,
    'feature_columns': feature_columns,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'identifier_columns': identifier_columns,
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'best_model_name': best_model_name,
    'model_selection_metric': 'average_precision_then_roc_auc',
    'results_path': str(RESULTS_PATH),
    'confusion_matrix_path': str(CONFUSION_PATH),
    'classification_report_path': str(CLASSIFICATION_REPORT_PATH),
    'best_model_path': str(BEST_MODEL_PATH),
}

with open(METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=4)

print(f'Best baseline model: {best_model_name}')
print(f'Results saved to: {RESULTS_PATH}')
print(f'Confusion matrices saved to: {CONFUSION_PATH}')
print(f'Classification reports saved to: {CLASSIFICATION_REPORT_PATH}')
print(f'Metadata saved to: {METADATA_PATH}')
print(f'Best model saved to: {BEST_MODEL_PATH}')

Best baseline model: hist_gradient_boosting
Results saved to: C:\Users\aitmo\financial-loan-analytics\reports\baseline_model_results.csv
Confusion matrices saved to: C:\Users\aitmo\financial-loan-analytics\reports\baseline_confusion_matrices.csv
Classification reports saved to: C:\Users\aitmo\financial-loan-analytics\reports\baseline_classification_report.csv
Metadata saved to: C:\Users\aitmo\financial-loan-analytics\reports\baseline_modeling_metadata.json
Best model saved to: C:\Users\aitmo\financial-loan-analytics\models\baseline_best_model.joblib


## 12. Baseline modeling summary

This summary closes the first modeling benchmark. Further work should focus on better validation, stronger models, threshold tuning, calibration, explainability, and feature improvements.

In [12]:
summary_lines = []

summary_lines.append(f'- Modeling rows used: **{len(model_df):,}**.')
summary_lines.append(f'- Train rows: **{len(X_train):,}**; test rows: **{len(X_test):,}**.')
summary_lines.append(f'- Target column: **`{target_column}`**.')
summary_lines.append(f'- Features used: **{len(feature_columns)}**.')
summary_lines.append(f'- Models trained: **{len(trained_models)}**.')
summary_lines.append(f'- Best baseline model by average precision: **`{best_model_name}`**.')

best_row = results_df.iloc[0]
summary_lines.append(
    f'- Best model ROC-AUC: **{best_row["roc_auc"]:.4f}**; average precision: **{best_row["average_precision"]:.4f}**.'
)
summary_lines.append(
    '- Imputation and encoding were performed inside model pipelines to reduce leakage risk.'
)
summary_lines.append(
    '- Next step: model improvement with time-aware validation, threshold tuning, calibration, and explainability.'
)

display(Markdown(chr(10).join(summary_lines)))

- Modeling rows used: **563,510**.
- Train rows: **450,808**; test rows: **112,702**.
- Target column: **`has_default_within_12_months`**.
- Features used: **27**.
- Models trained: **3**.
- Best baseline model by average precision: **`hist_gradient_boosting`**.
- Best model ROC-AUC: **0.8793**; average precision: **0.5896**.
- Imputation and encoding were performed inside model pipelines to reduce leakage risk.
- Next step: model improvement with time-aware validation, threshold tuning, calibration, and explainability.